# Day 7 — SQL & SQLAlchemy Basics

---

So far our data has lived in Python lists and JSON files. That breaks down the moment we have:

- More than one process touching the same data (**concurrency**)
- Anything more than a linear scan (**querying / indexing**)
- Rules like "email must be unique" (**integrity**)
- A million rows (**performance**)

A real **relational database** solves all of these. Today we'll meet SQL, then drive it from Python via the stdlib `sqlite3` module, and finally swap to **SQLAlchemy 2.0** — the typed ORM you'll use in real apps.

In [ ]:
!pip install sqlalchemy
# sqlite3 is part of the Python stdlib — nothing to install

## Relational basics

A relational database stores data in **tables** (think: spreadsheets with strict types).

| Concept | Meaning |
|---|---|
| **Table** | A collection of rows (e.g. `users`) |
| **Row** | One record (one user) |
| **Column** | A typed field (`name TEXT`, `age INTEGER`) |
| **Primary key (PK)** | A column that uniquely identifies a row (usually `id`) |
| **Foreign key (FK)** | A column that points to a PK in another table |
| **Constraint** | A rule the DB enforces (`NOT NULL`, `UNIQUE`, `CHECK`) |
| **Index** | A lookup structure that makes `WHERE` fast |

## SQL primer

SQL is the language databases speak. The four core verbs:

| Statement | Purpose | Example |
|---|---|---|
| `CREATE TABLE` | Define a table | `CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)` |
| `INSERT INTO` | Add rows | `INSERT INTO users (name) VALUES ('Alice')` |
| `SELECT` | Read rows | `SELECT * FROM users WHERE id = 1` |
| `UPDATE` | Change rows | `UPDATE users SET name='Bob' WHERE id=1` |
| `DELETE` | Remove rows | `DELETE FROM users WHERE id=1` |

Let's run each one with Python's built-in `sqlite3`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")  # in-memory DB, lives only for this kernel
cur = conn.cursor()

# CREATE
cur.execute("""
    CREATE TABLE users (
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        name  TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        age   INTEGER
    )
""")

# INSERT (always use ? placeholders — never f-strings — to avoid SQL injection)
cur.execute("INSERT INTO users (name, email, age) VALUES (?, ?, ?)", ("Alice", "a@x.com", 30))
cur.execute("INSERT INTO users (name, email, age) VALUES (?, ?, ?)", ("Bob",   "b@x.com", 17))
cur.execute("INSERT INTO users (name, email, age) VALUES (?, ?, ?)", ("Carol", "c@x.com", 42))
conn.commit()

# SELECT
for row in cur.execute("SELECT id, name, age FROM users"):
    print(row)

### `WHERE`, `ORDER BY`, `LIMIT`

Filtering and shaping the result set.

In [ ]:
# WHERE — only adults, ORDER BY age DESC, LIMIT 2
rows = cur.execute(
    "SELECT name, age FROM users WHERE age >= ? ORDER BY age DESC LIMIT ?",
    (18, 2),
).fetchall()
print(rows)

### `UPDATE` and `DELETE`

In [ ]:
cur.execute("UPDATE users SET age = ? WHERE name = ?", (31, "Alice"))
cur.execute("DELETE FROM users WHERE name = ?", ("Bob",))
conn.commit()

print(cur.execute("SELECT * FROM users").fetchall())

## `JOIN` basics

Real data is split across multiple tables. A **JOIN** stitches rows together by matching keys.

| Kind | Meaning |
|---|---|
| `INNER JOIN` | Only rows that match in **both** tables |
| `LEFT JOIN` | All rows from the left table + matches from the right (`NULL` if none) |
| `RIGHT JOIN` | Mirror of LEFT (not supported in SQLite) |
| `FULL OUTER JOIN` | Both sides, matched where possible (not in SQLite) |

In [ ]:
# Two related tables: users and their posts
cur.execute("CREATE TABLE posts (id INTEGER PRIMARY KEY, title TEXT, user_id INTEGER REFERENCES users(id))")
cur.executemany(
    "INSERT INTO posts (title, user_id) VALUES (?, ?)",
    [("Alice's first post", 1), ("Alice on databases", 1), ("Orphan post", 99)],
)
conn.commit()

print("INNER JOIN — only users WITH posts:")
for row in cur.execute("""
    SELECT users.name, posts.title
    FROM users
    INNER JOIN posts ON posts.user_id = users.id
"""):
    print(" ", row)

print("\nLEFT JOIN — every user, posts where they exist:")
for row in cur.execute("""
    SELECT users.name, posts.title
    FROM users
    LEFT JOIN posts ON posts.user_id = users.id
"""):
    print(" ", row)

conn.close()

## Indexes — what & when

An **index** is a sorted lookup structure (usually a B-tree) on one or more columns.

- `SELECT * FROM users WHERE email = ?` is **O(n)** without an index — every row gets scanned.
- With `CREATE INDEX ix_users_email ON users(email)` it becomes **O(log n)**.

Rules of thumb:

- Add indexes on columns you **filter by** or **join on**.
- `PRIMARY KEY` and `UNIQUE` columns are indexed automatically.
- Indexes cost disk space and slow down writes — don't index everything.

## ORM vs raw SQL

| | Raw SQL (`sqlite3`) | ORM (SQLAlchemy) |
|---|---|---|
| Boilerplate | Lots of string SQL | Python classes |
| Type safety | None | `Mapped[int]`, `Mapped[str]` |
| Refactoring | Search-and-replace SQL strings | Rename a field, IDE follows |
| Power user queries | Trivial | Possible, sometimes awkward |
| Learning curve | Low (if you know SQL) | Higher |

In production you'll almost always pick an ORM for the typical 90% of queries and drop down to raw SQL for the spicy 10%. From here on we use **SQLAlchemy 2.0**.

## SQLAlchemy 2.0 setup

Three pieces you always need:

1. An **engine** — knows how to talk to a database URL.
2. A **`Base`** class — root of your model hierarchy.
3. A **session factory** — short-lived units of work.

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker

# echo=True prints every SQL statement — handy while learning
engine = create_engine("sqlite:///./day7_demo.db", echo=False)

class Base(DeclarativeBase):
    pass

SessionLocal = sessionmaker(bind=engine)
print("engine:", engine.url)

## Defining a model

The SQLAlchemy 2.0 typed style uses `Mapped[...]` and `mapped_column(...)`. The types are real Python annotations — your IDE will autocomplete them.

In [ ]:
class User(Base):
    __tablename__ = "users"

    id:    Mapped[int] = mapped_column(primary_key=True)
    name:  Mapped[str]
    email: Mapped[str] = mapped_column(unique=True)

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r}, email={self.email!r})"

# Create the table (no-op if it already exists)
Base.metadata.create_all(engine)
print("tables:", list(Base.metadata.tables))

## CRUD with sessions

A `Session` is a **unit of work**: you stage changes, then `commit()`. Open one per logical operation (per HTTP request, per script run, etc.).

In [ ]:
# CREATE
with SessionLocal() as session:
    session.add(User(name="Alice", email="alice@example.com"))
    session.add(User(name="Bob",   email="bob@example.com"))
    session.commit()

In [ ]:
# READ — by primary key
from sqlalchemy import select

with SessionLocal() as session:
    alice = session.get(User, 1)
    print("by id  ->", alice)

    # READ — by filter (2.0 style: select().where() — NOT session.query())
    stmt = select(User).where(User.name == "Alice")
    result = session.execute(stmt).scalars().all()
    print("by name->", result)

    # READ — all
    all_users = session.execute(select(User)).scalars().all()
    print("all    ->", all_users)

In [ ]:
# UPDATE — mutate the object, commit
with SessionLocal() as session:
    user = session.get(User, 1)
    user.name = "Alice Cooper"
    session.commit()
    print("after update ->", session.get(User, 1))

In [ ]:
# DELETE
with SessionLocal() as session:
    bob = session.execute(select(User).where(User.name == "Bob")).scalar_one()
    session.delete(bob)
    session.commit()

    remaining = session.execute(select(User)).scalars().all()
    print("remaining ->", remaining)

> ⚠️ **2.0 style only.** You'll see `session.query(User).filter(...).all()` in older tutorials — that's the legacy 1.x API. Stick to `session.execute(select(User).where(...)).scalars().all()`. It's clearer, typed, and works the same for async.

## Recap

- A **relational DB** gives you concurrency, integrity, and fast querying for free.
- **SQL** has four core verbs: `CREATE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE`. Filter with `WHERE`, shape with `ORDER BY` / `LIMIT`, combine tables with `JOIN`.
- Use **parameterised queries** (`?` placeholders) — never f-string user input into SQL.
- **Indexes** make `WHERE` and `JOIN` fast. PKs and `UNIQUE` columns get one automatically.
- **SQLAlchemy 2.0** wraps all of that in typed Python: `DeclarativeBase`, `Mapped`, `mapped_column`, `select()`.
- A **session** is a unit of work: `add` / `get` / `execute(select(...))` / `delete`, then `commit`.

Next: relationships between tables, cascades, migrations, and wiring all of this into FastAPI.